# Monitoreo agroclimático de Honduras — FAO GIEWS / ASIS

Cuaderno de respaldo de **Datos agroclimáticos para Honduras**. Reconstruye desde
las fuentes primarias de FAO un panel municipio × dekad de indicadores
agroclimáticos y produce las once figuras del documento, en el mismo orden y sin
figuras adicionales.

| | |
|---|---|
| **Fuentes** | ImageServer ASIS (rásteres ~1 km) y CSV oficiales del portal GIEWS |
| **Unidad de análisis** | Municipio (GAUL 2015 nivel 2) |
| **Resolución temporal** | Dekad (tercios de mes) |
| **Periodo histórico** | Desde 2005, gobernado por la constante `START_YEAR` |
| **Series** | ASI, VCI, precipitación observada y anomalía de precipitación |

## Las once figuras

| Parte | Figuras |
|---|---|
| 1. Sequía de 2019 | 1.1 la temporada primera de 2019 frente a la norma histórica · 1.2 ASI por municipio · 1.3 cómo escaló el estrés, municipio por municipio |
| 2. Eta e Iota, 2020 | 2.1 la serie de lluvia · 2.2 los índices de sequía no miden una inundación · 2.3 VCI por municipio · 2.4 propagación y persistencia del daño |
| 3. Últimos 18 meses | 3.1 climatología del estrés agrícola · 3.2 los tres indicadores sobre el mismo eje · 3.3 ASI municipal · 3.4 VCI municipal |

## Antes de ejecutar

Se ejecuta de principio a fin y en orden: cada celda usa objetos definidos en las
anteriores. La primera corrida descarga varios miles de rásteres recortados y
tarda; todo queda en `./asis_cache`, de modo que las siguientes son rápidas y no
vuelven a golpear los servicios de FAO. Si `www.fao.org` responde 403, el bloqueo
viene de Cloudflare: el cuaderno indica en pantalla cómo colocar el CSV a mano.

Las figuras se generan con Plotly y son interactivas. En un visor estático no se
ven. El cuaderno se distribuye sin salidas guardadas, así que hay que ejecutarlo
para verlas.

Convención del código: los identificadores están en inglés y el texto —comentarios,
títulos y subtítulos de las figuras— en español.


## Cómo leer los indicadores

**Dekad.** ASIS trabaja en tercios de mes: D1 son los días 1 a 10, D2 del 11 al 20
y D3 del 21 al fin de mes. El año tiene 36 dekads. La notación del cuaderno es
`2019-09-D2`.

**ASI (Agricultural Stress Index).** Porcentaje del área de cultivo de la unidad
que estuvo bajo estrés hídrico durante la temporada. Es acumulativo dentro de la
temporada y se reinicia con la siguiente. Mide déficit: por construcción no puede
detectar un exceso de agua. Va de 0 a 100.

**VCI (Vegetation Condition Index).** Posición del vigor de la vegetación
observada frente a su propio historial reciente, de 0 a 1. Cubre todo el
territorio y todo el año, también fuera del área de cultivo. El umbral de alerta
de FAO es 0,35.

**Temporadas agrícolas.** La primera se siembra entre mayo y junio y se cosecha
entre agosto y septiembre (GS1); la postrera se siembra en septiembre y se cosecha
entre diciembre y enero (GS2). Fuera de su ventana, el ASI de esa temporada no
existe.

**Municipios en blanco.** No son municipios sin estrés: son municipios sin dato en
ese dekad, casi siempre por estar fuera de su ventana de cultivo. Los rásteres
codifican esas situaciones con banderas 251 a 255, que aquí se enmascaran en el
servidor para que nunca entren como valores del índice.


---

# 0. Infraestructura

Una celda: se importa el paquete `asis`, que trae el cliente de los servicios de
FAO, las geometrías GAUL con la estadística zonal y la capa de visualización. El
cálculo vive una sola vez ahí y no duplicado en cada cuaderno: una corrección en
la estadística zonal se hace en un solo lugar.

El periodo histórico arranca en 2005 y está gobernado por `START_YEAR`: la serie
de lluvia de GIEWS tiene un quiebre de homogeneidad alrededor de ese año, así que
encadenarla con los años previos introduce un salto que no es climático.

Para instalar las dependencias: `pip install -r requirements-build.txt`.


In [ ]:
# CELDA 1 · Infraestructura: se importa del paquete
#
# El cálculo vive una sola vez, en `asis/`. Este cuaderno lo importa, de modo
# que una corrección en la estadística zonal se hace en un solo lugar y no en
# dos.
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "asis").exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from asis import client, panel
from asis.config import (ASI_THRESHOLDS, CACHE, CLASSES, FLAGS, GRID_STEP,
                         HND_BBOX, NODATA, PALETTE, PIX_DEG, PIX_KM2, SEASONS,
                         SEASON_WINDOW, SERIES, SOURCE_NOTE, START_YEAR,
                         VALID_RANGE, WKID, in_season)
from asis.calendar import (MONTH_ES, dekad_between, dekad_code, dekad_date,
                           dekad_label, dekad_of_year, dekad_range,
                           dekad_window)
from asis.client import (SNAP, catalog_parsed, clip_period, export_tif,
                         export_tifs, last_dekad, load_csv, raster_name)
from asis.zonal import (SHAPE, TRANSFORM, export_geometry, grid,
                        municipal_series, read_tif, zonal_stats)
from asis.aggregate import (classify, climatology, department_weights,
                            national_from_csv, severity_area, to_country,
                            to_department, worst_case)
from asis.viz import (SCALE_ASI, SCALE_VCI, class_map, climatology_fig,
                      climatology_matrix, continuous_map, dashboard_fig,
                      heatmap_panel, rainfall_fig, ranking_fig, raster_detail,
                      series_fig, severity_area_fig, style_fig)

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 170)

# La malla de zonas se arma una vez y la reutilizan todos los dekads: es lo que
# hace viable recorrer veinte años de rásteres.
G = grid()
MUNI, DEPT, NZ, ZONES = G.muni, G.dept, G.nz, G.zones

# Geometría simplificada para las coropletas (~400 m: mantiene la forma y pesa
# unas diez veces menos que la original).
GEOJSON, MUNI_REF = export_geometry()

print("infraestructura lista ·", pd.Timestamp.today().date(),
      "· periodo desde", START_YEAR, "·", NZ, "municipios ·",
      len(DEPT), "departamentos")
print("último dekad publicado · ASI_D GS1 LC-C:",
      last_dekad("ASI_D", season="GS1", landcover="C"),
      "· VCI_D:", last_dekad("VCI_D"))


### 0.1 Series nacionales oficiales

Pondera la serie departamental que publica GIEWS por el área de cultivo real de
cada departamento —píxeles válidos del ráster, no número de departamentos— y arma
`asi_history`, que es la base de las figuras 1.1 y 3.1.


In [ ]:
# Series nacionales oficiales de GIEWS, ponderadas por área de cultivo
REF_DEKAD, CASE_YEAR = "2019-09-D2", 2019

# El peso de cada departamento es su área de cultivo en píxeles válidos del
# ráster. Ponderar por número de departamentos le daría a Islas de la Bahía el
# mismo peso que a Olancho.
DEPT_WEIGHT = department_weights(municipal_series(SERIES["asi_gs1"],
                                                 [REF_DEKAD]))

# Serie nacional a partir del CSV departamental de GIEWS: es el dato de FAO, no
# un agregado propio, y es la referencia contra la cual se valida el panel.
_asi_csv = load_csv("asi_dekad_s1")
_asi_csv = _asi_csv[_asi_csv["Land_Type"].astype(str)
                    .str.contains("Crop", case=False, na=False)]
asi_history = national_from_csv(clip_period(_asi_csv), DEPT_WEIGHT)

print(f"serie nacional ASI · {len(asi_history):,} dekads · "
      f"{int(asi_history['Year'].min())}-{int(asi_history['Year'].max())}")


### 0.2 Validación contra el dato oficial de FAO

Compara el agregado municipal propio contra la serie departamental que publica
GIEWS, sobre dos temporadas separadas en el tiempo (2019 y 2023): correlación,
error absoluto medio y sesgo. Es la única celda del cuaderno que no alimenta
ninguna figura; puede saltarse y el resto corre igual, pero conviene volver a
ejecutarla antes de citar cualquier cifra fuera del cuaderno.


In [ ]:
# CELDA 6 · Validación municipio -> departamento contra el CSV oficial
# Muestra independiente: dos temporadas de primera separadas en el tiempo.
DK_VALID = (dekad_range(2019, 6, 1, 2019, 10, 1)
            + dekad_range(2023, 6, 1, 2023, 9, 3))
valid_muni = municipal_series(SERIES["asi_gs1"], DK_VALID)
valid_dept = to_department(valid_muni)

official = load_csv("asi_dekad_s1")
official = official[official["Land_Type"].astype(str)
                    .str.contains("Crop", case=False, na=False)]
official = (official[["ADM1_CODE", "Province", "dekad_id", "Data"]]
            .dropna(subset=["Data"]))
official["adm1_code"] = official["ADM1_CODE"].astype("int64").astype(str)

check = valid_dept.merge(
    official[["adm1_code", "Province", "dekad_id", "Data"]],
    on=["adm1_code", "dekad_id"], how="inner")
check = check.rename(columns={"mean": "asi_raster", "Data": "asi_official"})
check["diff"] = check["asi_raster"] - check["asi_official"]

corr = np.corrcoef(check["asi_raster"], check["asi_official"])[0, 1]
mae = check["diff"].abs().mean()
bias = check["diff"].mean()
print(f"validación · n={len(check)} pares departamento-dekad · "
      f"r={corr:.3f} (R2={corr ** 2:.3f}) · MAE={mae:.2f} pp · "
      f"sesgo={bias:+.2f} pp")


---

# 1. Sequía agrícola de 2019 (El Niño)

El fenómeno de El Niño se refleja con claridad en la serie del ASI. En 2019 el
índice despega en el segundo dekad de julio y se mantiene elevado durante el resto
de la temporada.

La celda siguiente descarga los rásteres de los dieciocho dekads de la temporada
primera, arma el panel municipio × dekad e identifica el pico nacional.


In [ ]:
# CELDA 7 · Panel municipal de la temporada primera de 2019
DK_2019 = dekad_range(2019, 5, 1, 2019, 10, 3)
asi_2019 = municipal_series(SERIES["asi_gs1"], DK_2019)
asi_2019.to_parquet(CACHE / "panel_asi_2019.parquet")

country_2019 = to_country(asi_2019)
PEAK_2019 = country_2019.loc[country_2019["mean"].idxmax(), "dekad_id"]

_above_alert = country_2019[country_2019["mean"] >= 10]
ALERT_2019 = (_above_alert.iloc[0]["dekad_id"] if len(_above_alert)
              else DK_2019[0])
print(f"pico nacional: {dekad_label(PEAK_2019)} · "
      f"ASI {country_2019['mean'].max():.1f}%")


### 1.1 La temporada primera de 2019 frente a la norma histórica

La franja azul representa el rango habitual, calculado a partir de los catorce años
comprendidos entre 2005 y 2018.


In [ ]:
# FIGURA 1.1 · 2019 contra su propia historia
season_hist = asi_history[asi_history["dekad_of_year"].between(13, 30)]  # may-oct
baseline = season_hist[season_hist["Year"].between(START_YEAR, CASE_YEAR - 1)]
N_BASELINE = int(baseline["Year"].nunique())   # años que sostienen la línea base
pctl = baseline.groupby("dekad_of_year")["value"].quantile([.1, .5, .9]).unstack()
pctl.columns = ["p10", "p50", "p90"]
pctl = pctl.reset_index()
pctl["label"] = [f"{MONTH_ES[(k - 1) // 3 + 1]} D{(k - 1) % 3 + 1}"
                 for k in pctl["dekad_of_year"]]

fig_1_1 = go.Figure()
fig_1_1.add_scatter(x=pctl["label"], y=pctl["p90"],
                    name=f"p90 histórico ({START_YEAR}-{CASE_YEAR - 1})",
                    line=dict(width=0, color="#c9d6e3"), showlegend=False)
fig_1_1.add_scatter(x=pctl["label"], y=pctl["p10"],
                    name=f"rango p10-p90 ({START_YEAR}-{CASE_YEAR - 1})",
                    fill="tonexty", fillcolor="rgba(11,111,164,.15)",
                    line=dict(width=0, color="#c9d6e3"))
fig_1_1.add_scatter(x=pctl["label"], y=pctl["p50"], name="mediana histórica",
                    line=dict(color="#0b6fa4", width=2, dash="dot"))
for year, color in ((2019, "#b0413e"), (2023, "#e07b39"), (2025, "#7a5195")):
    s = season_hist[season_hist["Year"] == year]
    if len(s):
        fig_1_1.add_scatter(
            x=[f"{MONTH_ES[(k - 1) // 3 + 1]} D{(k - 1) % 3 + 1}"
               for k in s["dekad_of_year"]],
            y=s["value"], name=str(year), mode="lines+markers",
            line=dict(color=color, width=2.6))
fig_1_1.update_layout(height=470,
                      yaxis_title="ASI nacional (% del área de cultivo)")
style_fig(fig_1_1, "La temporada primera de 2019 frente a la norma histórica",
          f"Franja azul: rango habitual (p10-p90, {START_YEAR}-{CASE_YEAR - 1}, "
          f"n={N_BASELINE} años). 2019 se despega desde el segundo dekad de julio "
          "y no vuelve al rango hasta octubre.",
          y_source=-0.24)
fig_1_1.show()


### 1.2 Sequía agrícola: temporada primera de 2019, ASI por municipio

En el pico del evento, el tercer dekad de septiembre, los municipios más afectados
se encuentran en el centro-oriente del país y no necesariamente en el corredor
seco del suroeste. Los municipios en blanco no cuentan con área de cultivo con
información disponible para ese dekad.


In [ ]:
# FIGURA 1.2 · mapa animado de clases
fig_1_2 = class_map(asi_2019, GEOJSON, "ASI", "Sequía agrícola, temporada primera 2019 · ASI por municipio", "Cada cuadro es un dekad. El foco está en el oriente (Olancho y El Paraíso) "
    "y no en el corredor seco del suroeste. Los municipios en blanco no tienen "
    "área de cultivo con dato en ese dekad.", hover_extra={"pct_gt40": ":.0f", "p90": ":.0f"})
fig_1_2.show()


### 1.3 Cómo escaló el estrés, municipio por municipio

Entre los treinta municipios más afectados durante el pico, el estrés agrícola no
necesariamente aumentó de forma repentina: en varios casos se observa una
acumulación progresiva. La línea de referencia marca el dekad en el que el ASI
nacional superó el 10 %.


In [ ]:
# FIGURA 1.3 · escalada municipio x dekad
fig_1_3 = heatmap_panel(
    asi_2019, "mean",
    "Cómo escaló el estrés, municipio por municipio",
    f"30 municipios más afectados en el pico ({dekad_label(PEAK_2019)}). "
    f"La línea marca el dekad en que el ASI nacional superó 10 % "
    f"({dekad_label(ALERT_2019)}): ahí ya había señal para una alerta temprana.",
    family="ASI", top=100, ref_dekad=PEAK_2019, label="ASI %")
fig_1_3.add_vline(x=dekad_label(ALERT_2019),
                  line=dict(color="#111", width=1.6, dash="dot"))
fig_1_3.show()


---

# 2. Huracanes Eta e Iota, noviembre de 2020

Para analizar eventos asociados con exceso de lluvia, GIEWS distribuye además
precipitación acumulada en milímetros y anomalías de lluvia. El caso está aquí por
una razón metodológica: el ASI mide déficit hídrico y por construcción no puede
ver un exceso de agua.

La celda siguiente descarga los paneles municipales de VCI y de ASI de la postrera
para la ventana de octubre de 2020 a febrero de 2021.


In [ ]:
# CELDA 8 · Precipitación extrema: Eta e Iota (noviembre de 2020)
DK_ETA = dekad_range(2020, 10, 1, 2021, 2, 3)          # 15 dekads
vci_2020 = municipal_series(SERIES["vci"], DK_ETA)
asi_2020 = municipal_series(SERIES["asi_gs2"], dekad_range(2020, 9, 1, 2021, 1, 3))
vci_2020.to_parquet(CACHE / "panel_vci_2020.parquet")
asi_2020.to_parquet(CACHE / "panel_asi_2020_gs2.parquet")

BEFORE, IMPACT = "2020-10-D3", "2020-11-D2"
country_vci_2020 = to_country(vci_2020)
country_asi_2020 = to_country(asi_2020)


### 2.1 Eta e Iota en la serie de lluvia de FAO GIEWS

Lluvia dekadal nacional ponderada por área de cultivo, contra el promedio histórico
que publica FAO.


In [ ]:
# FIGURA 2.1 · lluvia nacional observada contra su promedio histórico
# Lluvia dekadal nacional y su promedio de largo plazo (LTA), ponderando
# departamentos por área de cultivo. La LTA es la que publica FAO y NO se
# recalcula: es la referencia oficial contra la que compara GIEWS.
rainfall = national_from_csv(clip_period(load_csv("rain_dekad")), DEPT_WEIGHT,
                             lta_col="Data_long_term_Average")
rainfall = rainfall.rename(columns={"value": "obs"})
window = rainfall[rainfall["dekad_id"].isin(
    dekad_range(2020, 8, 1, 2021, 2, 3))].copy()
window["label"] = [dekad_label(c) for c in window["dekad_id"]]

fig_2_1 = make_subplots(specs=[[{"secondary_y": True}]])
fig_2_1.add_bar(x=window["label"], y=window["obs"], name="lluvia observada",
                marker_color="#3b7dd8", opacity=0.9,
                hovertemplate="%{x}<br>%{y:.0f} mm<extra></extra>")
fig_2_1.add_scatter(x=window["label"], y=window["lta"],
                    name="promedio histórico (LTA)", mode="lines+markers",
                    line=dict(color="#0b3d91", dash="dot"),
                    hovertemplate="%{x}<br>LTA %{y:.0f} mm<extra></extra>")
fig_2_1.add_scatter(x=window["label"], y=window["anom_pct"],
                    name="anomalía (%)", mode="lines",
                    line=dict(color="#b0413e", width=2), secondary_y=True,
                    hovertemplate="%{x}<br>anomalía %{y:+.0f}%<extra></extra>")
fig_2_1.add_vrect(x0="1a dek nov 2020", x1="2a dek nov 2020", line_width=0,
                  fillcolor="#b0413e", opacity=0.10, layer="below")
fig_2_1.add_annotation(x="2a dek nov 2020", y=1.02, yref="paper",
                       showarrow=False,
                       text="Eta (3-5 nov) e Iota (16-18 nov)",
                       font=dict(size=11, color="#b0413e"))
fig_2_1.update_yaxes(title_text="mm por dekad", secondary_y=False)
fig_2_1.update_yaxes(title_text="anomalía sobre la LTA (%)", secondary_y=True,
                     showgrid=False)
fig_2_1.update_layout(height=470, xaxis=dict(tickangle=-45))
style_fig(fig_2_1, "Eta e Iota en la serie de lluvia de FAO GIEWS",
          "Lluvia dekadal nacional ponderada por área de cultivo, agosto de 2020 "
          "a febrero de 2021, contra el promedio histórico 1989-2015.",
          y_source=-0.30)
fig_2_1.show()

rain_peak = window.loc[window["anom_pct"].idxmax()]
print(f"máxima anomalía de lluvia: {dekad_label(rain_peak['dekad_id'])} · "
      f"{rain_peak['obs']:.0f} mm frente a {rain_peak['lta']:.0f} mm de LTA "
      f"({rain_peak['anom_pct']:+.0f}%)")


### 2.2 Los índices de sequía no son suficientes para medir una inundación

A escala nacional, tanto el VCI como el ASI muestran una capacidad limitada para
capturar el impacto agregado de Eta e Iota. El daño por exceso es local y solo
aparece al bajar al nivel municipal.


In [ ]:
# FIGURA 2.2 · el punto ciego del ASI
shared = sorted(set(country_vci_2020["dekad_id"])
                & set(country_asi_2020["dekad_id"]))
asi_vci = (country_vci_2020[country_vci_2020["dekad_id"].isin(shared)]
           [["dekad_id", "mean"]].rename(columns={"mean": "vci"})
           .merge(country_asi_2020[["dekad_id", "mean"]]
                  .rename(columns={"mean": "asi"}), on="dekad_id")
           .merge(rainfall[["dekad_id", "anom_pct"]], on="dekad_id", how="left"))
asi_vci["label"] = [dekad_label(c) for c in asi_vci["dekad_id"]]

fig_2_2 = make_subplots(specs=[[{"secondary_y": True}]])
fig_2_2.add_scatter(x=asi_vci["label"], y=asi_vci["vci"],
                    name="VCI nacional (0-1)", mode="lines+markers",
                    line=dict(color="#2f8f4e", width=2.8),
                    hovertemplate="%{x}<br>VCI %{y:.3f}<extra></extra>")
fig_2_2.add_scatter(x=asi_vci["label"], y=asi_vci["asi"],
                    name="ASI postrera (%)", mode="lines+markers",
                    line=dict(color="#c9a227", width=2.4, dash="dash"),
                    secondary_y=True,
                    hovertemplate="%{x}<br>ASI %{y:.2f}%<extra></extra>")
fig_2_2.add_hline(y=0.35, line=dict(color="#ff8900", width=1.2, dash="dot"))
fig_2_2.add_annotation(x=asi_vci["label"].iloc[0], y=0.35,
                       text="umbral FAO VCI 0,35", showarrow=False, yshift=10,
                       xanchor="left", font=dict(size=10, color="#ff8900"))
fig_2_2.update_yaxes(title_text="VCI (0-1)", secondary_y=False, range=[0, 1])
fig_2_2.update_yaxes(title_text="ASI temporada postrera (%)", secondary_y=True,
                     showgrid=False)
fig_2_2.update_layout(height=460, xaxis=dict(tickangle=-45))
_vci_before = country_vci_2020.set_index("dekad_id").loc[BEFORE, "mean"]
_vci_impact = country_vci_2020.set_index("dekad_id").loc[IMPACT, "mean"]
style_fig(fig_2_2,
          "Ningún índice de sequía sirve para medir una inundación",
          f"Con la anomalía de lluvia más alta de la serie, el ASI de la postrera "
          f"se movió dentro de su rango habitual (máximo {asi_vci['asi'].max():.1f} %) "
          f"y el VCI nacional incluso subió de {_vci_before:.2f} a "
          f"{_vci_impact:.2f}: más agua reverdece el promedio del país. El daño "
          "por exceso es local, hay que buscarlo por municipio y no en el "
          "agregado nacional.",
          y_source=-0.30)
fig_2_2.show()


### 2.3 Condición de la vegetación por municipio durante Eta e Iota

VCI dekadal de octubre de 2020 a febrero de 2021. La serie nacional prácticamente
no se mueve, pero un grupo de municipios del Valle de Sula, Santa Bárbara y el
occidente del país presenta una caída pronunciada durante el segundo dekad de
noviembre.


In [ ]:
# FIGURA 2.3 · mapa animado del deterioro vegetal
fig_2_3 = class_map(vci_2020, GEOJSON, "VCI", "Huracanes Eta e Iota · condición de la vegetación por municipio", "VCI dekadal de octubre de 2020 a febrero de 2021. Rojo: vegetación en peor "
    "estado que su historia reciente; verde: mejor. El promedio nacional no se "
    "mueve, pero un grupo de municipios del Valle de Sula, Santa Bárbara y el "
    "occidente cae con fuerza en el segundo dekad de noviembre.", hover_extra={"pct_lt0.35": ":.0f"})
fig_2_3.show()


### 2.4 Propagación y persistencia del daño

Los treinta municipios con peor condición de la vegetación durante el dekad de
impacto, seguidos en el tiempo: se observa el golpe de noviembre y cuánto tarda
cada municipio en recuperar niveles óptimos de VCI.


In [ ]:
# FIGURA 2.4 · propagación municipio x dekad
fig_2_4 = heatmap_panel(
    vci_2020, "mean",
    "Propagación y persistencia del daño",
    "30 municipios con menor VCI en el dekad de impacto. Se ve el golpe de "
    "noviembre y cuánto tarda cada municipio en volver al verde.",
    family="VCI", top=30, ref_dekad=IMPACT, value_range=(0, 1), label="VCI")
fig_2_4.show()


---

# 3. Últimos 18 meses publicados

La ventana no se fija a mano: se le pregunta al catálogo cuál es el último dekad
efectivamente publicado y se retroceden 54 dekads, de modo que esta parte sigue
siendo válida cuando FAO publique rásteres nuevos, sin editar nada.

Se descargan las dos temporadas, primera y postrera, porque el ASI solo tiene
sentido dentro de su ventana de cultivo. En los dekads en que ambas están activas
se conserva el peor caso vigente, criterio conservador para alerta temprana. Se
trae además el VCI, que sí es continuo todo el año. Es la celda más pesada del
cuaderno en la primera ejecución.


In [ ]:
# CELDA 9 · Paneles municipales de los últimos 18 meses
LAST = last_dekad("ASI_D", season="GS1", landcover="C")
DK_18M = dekad_window(LAST, 54)
print(f"ventana: {DK_18M[0]} a {DK_18M[-1]} ({len(DK_18M)} dekads · "
      f"{dekad_label(DK_18M[0])} - {dekad_label(DK_18M[-1])})\n")

asi_gs1 = municipal_series(SERIES["asi_gs1"], DK_18M)
asi_gs2 = municipal_series(SERIES["asi_gs2"], DK_18M)
vci_18m = municipal_series(SERIES["vci"], DK_18M)

# --- Unión de temporadas: se conserva el peor caso vigente --------------------
# En los dekads en que las dos temporadas están activas se toma el ASI máximo
# (criterio conservador para alerta temprana) y se deja registrado de cuál viene.
asi_18m = (pd.concat([asi_gs1, asi_gs2], ignore_index=True)
           .dropna(subset=["mean"])
           .sort_values(["adm2_code", "dekad_id", "mean"],
                        ascending=[True, True, False])
           .drop_duplicates(["adm2_code", "dekad_id"], keep="first")
           .reset_index(drop=True))
asi_18m["severity"] = classify(asi_18m["mean"], "ASI").astype(str)
asi_18m.to_parquet(CACHE / "panel_asi_18m.parquet")
vci_18m.to_parquet(CACHE / "panel_vci_18m.parquet")

country_18m = to_country(asi_18m)
country_18m_vci = to_country(vci_18m)
CURRENT = asi_18m[asi_18m["dekad_id"] == LAST].copy()
CURRENT_VCI = vci_18m[vci_18m["dekad_id"] == LAST]

rainfall_18m = rainfall[rainfall["dekad_id"].isin(DK_18M)].copy()
rainfall_18m["date"] = rainfall_18m["dekad_id"].map(dekad_date)

print(f"ASI nacional en {dekad_label(LAST)}: "
      f"{country_18m[country_18m['dekad_id'] == LAST]['mean'].iloc[0]:.2f}% · "
      f"VCI nacional: "
      f"{country_18m_vci[country_18m_vci['dekad_id'] == LAST]['mean'].iloc[0]:.3f}")


### 3.1 Climatología del estrés agrícola, 2005-2026

ASI nacional por dekad de la temporada primera, de mayo a octubre. Cada franja roja
horizontal es una sequía agrícola; la línea punteada marca 2019. La vista se limita
a la ventana de cultivo porque el ASI es acumulativo dentro de la temporada: fuera
de ella el valor está congelado y graficarlo sugeriría una persistencia que no
existe.


In [ ]:
# FIGURA 3.1 · climatología dekadal del periodo de análisis
clim_matrix = (asi_history[asi_history["dekad_of_year"].between(13, 30)]
               .pivot_table(index="Year", columns="dekad_of_year",
                            values="value"))
clim_labels = [f"{MONTH_ES[(k - 1) // 3 + 1]} D{(k - 1) % 3 + 1}"
               for k in clim_matrix.columns]

fig_3_1 = go.Figure(go.Heatmap(
    z=clim_matrix.values, x=clim_labels, y=clim_matrix.index.astype(int),
    colorscale=SCALE_ASI, zmin=0,
    zmax=float(np.nanpercentile(clim_matrix.values, 99.5)),
    xgap=0.5, ygap=0.5,
    colorbar=dict(title="ASI %", thickness=14, len=0.85),
    hovertemplate="%{y} · %{x}<br>ASI %{z:.1f}%<extra></extra>"))
fig_3_1.add_hline(y=CASE_YEAR, line=dict(color="#111", width=1.4, dash="dot"))
fig_3_1.update_layout(height=640, yaxis=dict(dtick=1, autorange="reversed"),
                      xaxis=dict(tickangle=-45))
style_fig(fig_3_1,
          f"Climatología del estrés agrícola, {int(clim_matrix.index.min())}-"
          f"{int(clim_matrix.index.max())}",
          "ASI nacional por dekad de la temporada primera (mayo-octubre), "
          "ponderado por área de cultivo de cada departamento. Cada franja roja "
          "es una sequía agrícola; la línea punteada marca 2019.",
          y_source=-0.13, legend="off")
fig_3_1.show()


### 3.2 Honduras en los últimos 18 meses

ASI nacional, VCI nacional y anomalía de lluvia sobre el mismo eje temporal: los
tres indicadores apuntan a la persistencia de condiciones agroclimáticas adversas.


In [ ]:
# FIGURA 3.2 · estrés, vegetación y lluvia en el mismo eje de tiempo
fig_3_2 = make_subplots(
    rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.07,
    subplot_titles=("ASI nacional · % del área de cultivo bajo estrés hídrico",
                    "VCI nacional · condición de la vegetación (0-1)",
                    "Anomalía de lluvia sobre el promedio histórico (%)"))
fig_3_2.add_scatter(x=country_18m["date"], y=country_18m["mean"], name="ASI",
                    mode="lines+markers", line=dict(color="#b0413e", width=2.4),
                    fill="tozeroy", fillcolor="rgba(176,65,62,.18)",
                    hovertemplate="%{x|%d %b %Y}<br>ASI %{y:.1f}%<extra></extra>",
                    row=1, col=1)
fig_3_2.add_scatter(x=country_18m_vci["date"], y=country_18m_vci["mean"],
                    name="VCI", mode="lines",
                    line=dict(color="#2f8f4e", width=2.4),
                    hovertemplate="%{x|%d %b %Y}<br>VCI %{y:.3f}<extra></extra>",
                    row=2, col=1)
fig_3_2.add_hline(y=0.35, line=dict(color="#ff8900", width=1.1, dash="dot"),
                  row=2, col=1)
fig_3_2.add_bar(x=rainfall_18m["date"], y=rainfall_18m["anom_pct"],
                name="anomalía de lluvia",
                marker_color=np.where(rainfall_18m["anom_pct"] >= 0,
                                      "#3b7dd8", "#c98b3b"),
                hovertemplate="%{x|%d %b %Y}<br>%{y:+.0f}%<extra></extra>",
                row=3, col=1)
fig_3_2.add_hline(y=0, line=dict(color="#666", width=1), row=3, col=1)
fig_3_2.update_layout(height=760, showlegend=False)
fig_3_2.update_annotations(font_size=12, x=0, xanchor="left")
style_fig(fig_3_2,
          f"Honduras en los últimos 18 meses · hasta {dekad_label(LAST)}",
          "Tres indicadores independientes sobre el mismo eje temporal: déficit "
          "hídrico acumulado en el cultivo (ASI), estado de la vegetación (VCI) "
          "y lluvia observada frente a su normal.",
          y_source=-0.11, legend="off")
fig_3_2.show()


### 3.3 ASI municipal en el último dekad publicado

Indicador estacional: considera la fenología y la máscara de cultivo. En blanco,
municipios fuera de temporada o sin área de cultivo, que no es lo mismo que sin
estrés.


In [ ]:
# FIGURA 1.3 · escalada municipio x dekad
fig_1_3 = heatmap_panel(
    CURRENT, "mean",
    "Cómo escaló el estrés, municipio por municipio",
    f"30 municipios más afectados en el pico ({dekad_label('2026-05-D1')}). "
    f"La línea marca el dekad en que el ASI nacional superó 10 % "
    f"({dekad_label('2026-05-D1')}): ahí ya había señal para una alerta temprana.",
    family="ASI", top=100, ref_dekad='2026-05-D1', label="ASI %")
fig_1_3.add_vline(x=dekad_label('2026-05-D1'),
                  line=dict(color="#111", width=1.6, dash="dot"))
fig_1_3.show()


In [ ]:
# FIGURA 3.3 · ASI municipal, foto del momento
fig_3_3 = class_map(CURRENT, GEOJSON, "ASI", f"Estado agroclimático de Honduras · {dekad_label(LAST)}", "ASI municipal en el último dekad publicado por FAO GIEWS. Verde: sin estrés "
    "por déficit hídrico; rojo: más del 70 % del área de cultivo del municipio "
    "bajo estrés. En blanco, municipios fuera de temporada o sin cultivo.", animation=None, hover_extra={"pct_gt40": ":.0f", "p90": ":.0f"})
fig_3_3.show()


### 3.4 VCI municipal en el último dekad publicado

Indicador no estacional: refleja la condición general de la vegetación, también
fuera de temporada y fuera del área de cultivo. Leer ASI y VCI juntos evita los dos
errores típicos: creer que no pasa nada donde el ASI no aplica, y confundir un
exceso de agua con una sequía.


In [ ]:
# FIGURA 3.4 · VCI municipal, el mismo dekad visto con el otro índice
fig_3_4 = class_map(CURRENT_VCI, GEOJSON, "VCI", f"Condición de la vegetación · {dekad_label(LAST)}", "El VCI cubre todo el territorio y todo el año, también fuera de temporada y "
    "fuera del área de cultivo. Leer ASI y VCI juntos evita los dos errores "
    "típicos: creer que no pasa nada donde el ASI no aplica, y confundir un "
    "exceso de agua con una sequía.", animation=None, hover_extra={"pct_lt0.35": ":.0f"})
fig_3_4.show()


---

## Exportación


In [ ]:
# CELDA 10 · Exportación de las once figuras y de los paneles
OUTDIR = Path("./salidas"); OUTDIR.mkdir(exist_ok=True)

figures = {"1_1_2019_vs_norma": fig_1_1,
           "1_2_asi_municipal_2019": fig_1_2,
           "1_3_escalada_municipal_2019": fig_1_3,
           "2_1_lluvia_eta_iota": fig_2_1,
           "2_2_punto_ciego_asi": fig_2_2,
           "2_3_vci_municipal_2020": fig_2_3,
           "2_4_propagacion_vci": fig_2_4,
           "3_1_climatologia_asi": fig_3_1,
           "3_2_tablero_18m": fig_3_2,
           "3_3_asi_municipal_actual": fig_3_3,
           "3_4_vci_municipal_actual": fig_3_4}
for name, fig in figures.items():
    fig.write_html(OUTDIR / f"{name}.html", include_plotlyjs="cdn")

tables = {"panel_asi_2019": asi_2019, "panel_vci_2020": vci_2020,
          "panel_asi_18m": asi_18m, "panel_vci_18m": vci_18m}
for name, table in tables.items():
    table.to_parquet(OUTDIR / f"{name}.parquet")
    table.to_csv(OUTDIR / f"{name}.csv", index=False)

print(f"exportado a {OUTDIR.resolve()} · {len(figures)} figuras HTML y "
      f"{len(tables)} tablas (parquet y csv)")


---

## Limitaciones que conviene declarar al publicar

El ASI solo existe dentro de la ventana de cultivo de cada temporada. Fuera de ella
el municipio aparece sin dato, no en cero, y tratarlo como cero fabrica una calma
que el índice no afirma.

El análisis histórico arranca en 2005 por el quiebre de homogeneidad de la serie de
lluvia. Eso deja la línea base de percentiles en catorce años: razonable para
mediana y cuartiles, débil para caracterizar extremos.

Los límites de GAUL 2015 no coinciden en todos los casos con la división
administrativa vigente. Los agregados municipales de este cuaderno son comparables
entre sí, pero no son cifras oficiales de ninguna unidad territorial.

Los índices miden condición de la vegetación y déficit hídrico, no producción ni
pérdida. El paso de estrés agroclimático a impacto en cosecha o en seguridad
alimentaria requiere información que no está aquí.

## Cómo extender el cuaderno

Para mover el periodo histórico basta cambiar `START_YEAR` en la celda 1. Para
cambiar de país se ajustan `ISO3`, `GAUL_COUNTRY` y `HND_BBOX` en la misma celda,
revisando las temporadas agrícolas, que son específicas de Honduras. Para agregar
un indicador se declara su rango válido en `VALID_RANGE` y se reutiliza
`municipal_series()`, que ya devuelve el panel municipio × dekad en formato largo.
